# Using ClusterIQ Self-Healing from Databricks Notebook

This notebook shows how to use the ClusterIQ POC to diagnose and fix notebook execution errors.

**Use Cases:**
- Diagnose RunExecutionError in jobs
- Auto-restart failed clusters
- Check cluster health status
- Get cost optimization recommendations
- Analyze job failures

## 1. Setup - ClusterIQ API Configuration

In [ ]:
import requests
import json
from datetime import datetime

# ClusterIQ Backend API URL (replace with your backend URL)
CLUSTERIQ_API = "http://localhost:8000"  # Change to your actual ClusterIQ backend URL

# Or if deployed to Azure Web App:
# CLUSTERIQ_API = "https://your-app.azurewebsites.net"

def call_api(endpoint, method="GET", payload=None):
    """Helper function to call ClusterIQ API"""
    url = f"{CLUSTERIQ_API}{endpoint}"
    try:
        if method == "GET":
            response = requests.get(url, timeout=30)
        elif method == "POST":
            response = requests.post(url, json=payload, timeout=30)
        elif method == "PUT":
            response = requests.put(url, json=payload, timeout=30)
        
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"❌ API Error: {e}")
        return None

print("✅ ClusterIQ API configured")

## 2. Check Cluster Health Status

First, check if your cluster has any health issues that might cause RunExecutionError

In [ ]:
# Get current cluster health status
health = call_api("/api/health")

if health:
    print("📊 CLUSTER HEALTH STATUS")
    print("=" * 70)
    
    summary = health.get('summary', {})
    print(f"\n🔍 Total Clusters: {summary.get('total_clusters', 0)}")
    print(f"✅ Healthy: {summary.get('healthy_clusters', 0)}")
    print(f"⚠️  Warning: {summary.get('warning_clusters', 0)}")
    print(f"❌ Failed: {summary.get('failed_clusters', 0)}")
    print(f"💤 Idle: {summary.get('idle_clusters', 0)}")
    
    # Show failed/warning clusters in detail
    if health.get('clusters'):
        print("\n🔥 CLUSTERS WITH ISSUES:")
        print("=" * 70)
        for cluster in health['clusters']:
            if cluster.get('status') in ['WARNING', 'FAILED', 'ERROR']:
                print(f"\n  Cluster: {cluster.get('cluster_name')}")
                print(f"  ID: {cluster.get('cluster_id')}")
                print(f"  Status: {cluster.get('status')} ({cluster.get('state')})")
                print(f"  Message: {cluster.get('message')}")
                if cluster.get('issues'):
                    print(f"  Issues: {', '.join(cluster['issues'])}")

## 3. Get Self-Healing Configuration

Check what self-healing features are enabled

In [ ]:
# Get self-healing config
config = call_api("/api/self-healing/config")

if config:
    cfg = config.get('config', {})
    print("⚙️  SELF-HEALING CONFIGURATION")
    print("=" * 70)
    print(f"\n🔧 Global Enabled: {cfg.get('enabled')}")
    print(f"🧪 Dry-Run Mode: {cfg.get('safety', {}).get('dry_run')}")
    
    features = cfg.get('features', {})
    print("\n📋 Features:")
    print(f"  ↻ Auto-restart failed clusters: {features.get('auto_restart_failed_clusters')}")
    print(f"  ⏹  Auto-terminate idle clusters: {features.get('auto_terminate_idle_clusters')}")
    print(f"  📈 Auto-scale adjustments: {features.get('auto_scale_adjustments')}")
    print(f"  ✨ Auto-apply optimizations: {features.get('auto_apply_optimizations')}")
    print(f"  🔍 Proactive health checks: {features.get('proactive_health_checks')}")

## 4. Enable Self-Healing (if not enabled)

Enable auto-restart for failed clusters to fix RunExecutionError

In [ ]:
# Enable self-healing with auto-restart
update_config = {
    "enabled": True,
    "features": {
        "auto_restart_failed_clusters": True,
        "auto_terminate_idle_clusters": True,
        "proactive_health_checks": True
    },
    "safety": {
        "dry_run": False  # Set to True for testing without actual changes
    }
}

result = call_api("/api/self-healing/config", method="PUT", payload=update_config)

if result:
    print("✅ Self-healing configuration updated successfully!")
    print(f"   Enabled: {result.get('config', {}).get('enabled')}")
    print(f"   Auto-restart: {result.get('config', {}).get('features', {}).get('auto_restart_failed_clusters')}")
else:
    print("❌ Failed to update configuration")

## 5. Run Self-Healing Now

Manually trigger self-healing to fix failed clusters immediately

In [ ]:
# Trigger self-healing
healing_result = call_api("/api/self-healing/run", method="POST")

if healing_result:
    print("🔄 SELF-HEALING EXECUTION RESULTS")
    print("=" * 70)
    
    summary = healing_result.get('summary', {})
    print(f"\n📊 Scan Summary:")
    print(f"   • Clusters scanned: {summary.get('scanned_clusters', 0)}")
    print(f"   • Running clusters: {summary.get('running_clusters', 0)}")
    print(f"   • Failed clusters: {summary.get('failed_clusters', 0)}")
    print(f"   • Actions taken: {summary.get('actions_taken', 0)}")
    
    # Show actions performed
    if healing_result.get('results'):
        print("\n✨ Actions Performed:")
        for action in healing_result['results']:
            status_icon = "✅" if action.get('success') else "❌"
            print(f"\n{status_icon} {action.get('action', 'Unknown action')}")
            print(f"   Cluster: {action.get('cluster_name')}")
            print(f"   Message: {action.get('message')}")
    
    # Show diagnostics
    if healing_result.get('diagnostics'):
        print("\n🔍 Diagnostics:")
        for diag in healing_result['diagnostics'][:5]:  # Show first 5
            print(f"\n   • Cluster: {diag.get('cluster_name')}")
            print(f"     Issue: {diag.get('issue')}")
            print(f"     Available: {diag.get('action_available')}")

## 6. Check Recent Healing History

View what self-healing actions were taken recently

In [ ]:
# Get healing history
history = call_api("/api/self-healing/history?limit=10")

if history:
    print("📜 RECENT SELF-HEALING ACTIONS")
    print("=" * 70)
    
    actions = history.get('history', [])
    if actions:
        for i, action in enumerate(actions, 1):
            status_icon = "✅" if action.get('status') == 'success' else "❌" if action.get('status') == 'failed' else "🔵"
            print(f"\n{i}. {status_icon} {action.get('action_type', 'Unknown').upper()}")
            print(f"   Cluster: {action.get('details', {}).get('cluster_name', 'N/A')}")
            print(f"   Status: {action.get('status')}")
            print(f"   Time: {action.get('timestamp')}")
            if action.get('details', {}).get('reason'):
                print(f"   Reason: {action['details']['reason']}")
    else:
        print("\nNo recent healing actions found.")

## 7. Get Recommendations for Your Cluster

See what optimizations ClusterIQ recommends

In [ ]:
# Get recommendations
recommendations = call_api("/api/approvals?status=")

if recommendations:
    print("💡 RECOMMENDATIONS")
    print("=" * 70)
    
    recs = recommendations.get('recommendations', [])
    if recs:
        for i, rec in enumerate(recs[:5], 1):  # Show first 5
            severity_icon = "🔴" if rec.get('severity') == 'HIGH' else "🟡" if rec.get('severity') == 'MEDIUM' else "🟢"
            print(f"\n{i}. {severity_icon} [{rec.get('severity')}] {rec.get('title')}")
            print(f"   {rec.get('description')}")
            print(f"   Status: {rec.get('status', 'PENDING')}")
            if rec.get('estimated_savings'):
                print(f"   💰 Savings: {rec.get('estimated_savings')}")
    else:
        print("\n✨ No recommendations - everything looks good!")

## 8. Diagnose Specific Job Failure

If you have a specific job with RunExecutionError, analyze it

In [ ]:
# Get jobs list to find failed jobs
jobs_data = call_api("/api/jobs")

if jobs_data:
    print("📊 JOBS STATUS")
    print("=" * 70)
    
    jobs = jobs_data.get('jobs', [])
    failed_jobs = [j for j in jobs if j.get('status') in ['FAILED', 'ERROR', 'TIMEOUT']]
    
    if failed_jobs:
        print(f"\n❌ Found {len(failed_jobs)} failed jobs:\n")
        for job in failed_jobs:
            print(f"   • Job: {job.get('name')}")
            print(f"     ID: {job.get('job_id')}")
            print(f"     Status: {job.get('status')}")
            print(f"     Last Run: {job.get('last_run_time', 'N/A')}")
            if job.get('error_message'):
                print(f"     Error: {job.get('error_message')}")
            print()
    else:
        print("\n✅ No failed jobs found!")

## 9. Analyze Your Notebook's Cluster

Get detailed info about the cluster running this notebook

In [ ]:
# Get current cluster info from Spark context
try:
    cluster_id = spark.conf.get("spark.databricks.clusterUsageTags.clusterId")
    cluster_name = spark.conf.get("spark.databricks.clusterUsageTags.clusterName")
    
    print(f"📍 Current Cluster: {cluster_name}")
    print(f"   ID: {cluster_id}")
    
    # Get this cluster's status from ClusterIQ
    health = call_api("/api/health")
    if health:
        clusters = health.get('clusters', [])
        current = next((c for c in clusters if c.get('cluster_id') == cluster_id), None)
        
        if current:
            print(f"\n📊 Health Status: {current.get('status')}")
            print(f"   State: {current.get('state')}")
            print(f"   Workers: {current.get('num_workers', 0)}")
            print(f"   Node Type: {current.get('node_type_id')}")
            
            if current.get('issues'):
                print(f"\n⚠️  Issues detected:")
                for issue in current['issues']:
                    print(f"   • {issue}")
except:
    print("⚠️  Not running in Databricks cluster or unable to get cluster info")

## 10. Quick Fix for RunExecutionError

Common fixes for notebook execution errors using ClusterIQ

In [ ]:
print("🔧 QUICK FIX CHECKLIST FOR RunExecutionError")
print("=" * 70)

# Step 1: Check cluster health
health = call_api("/api/health")
if health:
    failed = health.get('summary', {}).get('failed_clusters', 0)
    print(f"\n1. ✓ Cluster Health Check: {failed} failed clusters")
    if failed > 0:
        print("   → Action: Run self-healing to restart failed clusters")

# Step 2: Check self-healing config
config = call_api("/api/self-healing/config")
if config:
    enabled = config.get('config', {}).get('enabled')
    print(f"\n2. ✓ Self-Healing Enabled: {enabled}")
    if not enabled:
        print("   → Action: Enable self-healing to auto-fix cluster issues")

# Step 3: Run healing
print("\n3. Running self-healing now...")
healing = call_api("/api/self-healing/run", method="POST")
if healing:
    actions = healing.get('summary', {}).get('actions_taken', 0)
    print(f"   ✓ Self-Healing Complete: {actions} actions taken")

print("\n" + "=" * 70)
print("✅ Quick fix complete! Re-run your notebook now.")
print("\nIf issues persist:")
print("  1. Check the 'Recent Actions' section above")
print("  2. Review recommendations for your cluster")
print("  3. Check Databricks cluster logs for detailed errors")

## Summary

### What This Notebook Does:

1. **Health Check**: Identifies failed/unhealthy clusters
2. **Self-Healing**: Auto-restarts failed clusters
3. **Diagnostics**: Shows what issues were found and fixed
4. **Recommendations**: Suggests optimizations
5. **Job Analysis**: Finds failed jobs and their errors

### Common RunExecutionError Causes Fixed by ClusterIQ:

- ✅ Cluster in FAILED/ERROR state → **Auto-restart**
- ✅ Cluster terminated unexpectedly → **Auto-restart**
- ✅ Out of memory issues → **Recommendations for resize**
- ✅ Idle timeout → **Auto-termination config**
- ✅ Resource constraints → **Scaling recommendations**

### Next Steps:

1. Keep self-healing enabled for automatic fixes
2. Monitor the ClusterIQ dashboard at http://localhost:5173
3. Review and apply recommendations regularly
4. Check healing history to track all automatic actions